In [2]:
%load_ext autoreload
%autoreload 2

%autosave 30

Autosaving every 30 seconds


In [3]:
import copy

import torch
from torch.utils.data import DataLoader, random_split

from mermaidseg.datasets.dataset import CombinedCoralDataset, CoralNetDataset, MermaidDataset
from mermaidseg.io import get_parser, setup_config, update_config_with_args
from mermaidseg.logger import Logger

In [4]:
import os

os.environ["AWS_PROFILE"] = "mermaid-core"

## CoralNet annotations

Merged CoralNet point annotations are stored as a single Parquet file on S3 (see `CoralNetDataset` in `mermaidseg/datasets/dataset.py`). The block below loads that file with Ibis and inspects the S3 object layout for source `4957` (images are expected under `coralnet-public-images/s4957/images/`).

In [ ]:
import boto3
import ibis
import pandas as pd
from great_tables import GT
from ibis import _

# Defaults match CoralNetDataset in mermaidseg/datasets/dataset.py
CORALNET_SOURCE_BUCKET = "dev-datamermaid-sm-sources"
CORALNET_ANNOTATIONS_PARQUET = "coralnet_annotations_30112025.parquet"
CORALNET_ANNOTATIONS_URI = f"s3://{CORALNET_SOURCE_BUCKET}/{CORALNET_ANNOTATIONS_PARQUET}"
CORALNET_SOURCE_S3_PREFIX = "coralnet-public-images"
EXAMPLE_SOURCE_ID = 5086
AUDIT_PATH = "coralnet_source_audit_20260423.parquet"
AUDIT_URI = f"s3://{CORALNET_SOURCE_BUCKET}/dev/{AUDIT_PATH}"
ibis.options.interactive = True

con = ibis.duckdb.connect()
con.raw_sql(
    """
    INSTALL httpfs;
    LOAD httpfs;
    """
)
# Second statement: DuckDB validates the credential chain here — if this fails, fix AWS first
# (skill preflight: `aws sts get-caller-identity`, or rerun the notebook cell that runs check_aws_session).
con.raw_sql("CREATE OR REPLACE SECRET s3 (TYPE S3, PROVIDER CREDENTIAL_CHAIN)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [22]:
s3_audit = con.read_parquet(AUDIT_URI)

In [23]:
# Recompute is_complete so that it also includes that n_annotations > 0
s3_audit = s3_audit.mutate(
    is_complete=_.has_images_folder & _.has_annotations_csv & _.has_image_list_csv & _.has_annotations_csv & (_.n_annotations > 0)
)

In [24]:
s3_stats = s3_audit.describe().to_pandas()
s3_stats

,name,pos,type,count,nulls,unique,mean,std,min,p25,p50,p75,max
0,has_labelset_csv,5,boolean,1500,0,2,0.601333,NaN,NaN,NaN,NaN,NaN,NaN
1,has_metadata_csv,6,boolean,1500,0,2,0.408000,NaN,NaN,NaN,NaN,NaN,NaN
2,n_images_s3,7,int64,1500,0,502,734.047333,3519.275672,0.0,19.00,50.0,257.25,79062.0
3,n_images_csv,8,int64,1500,0,397,338.232667,1493.813652,0.0,0.00,10.0,105.50,32176.0
4,n_annotations,9,int64,1500,0,492,17875.164000,97947.577884,0.0,0.00,120.0,2400.00,2951200.0
5,n_unique_images_annotated,10,int64,1500,0,381,602.536667,3565.247014,0.0,0.00,4.0,89.25,79064.0
6,n_labels,11,float64,1500,598,134,46.771619,46.662685,1.0,15.00,25.0,75.00,273.0
7,n_metadata_rows,12,float64,1500,888,22,3.485294,3.761478,1.0,1.00,2.0,4.00,32.0
8,annotations_empty,13,boolean,1500,0,2,0.052667,NaN,NaN,NaN,NaN,NaN,NaN
9,image_list_empty,14,boolean,1500,0,1,0.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
summary_data = {
    "Metric": [
        "Total sources",
        "Complete sources",
        "Sources with images folder",
        "Sources with annotations.csv",
        "Sources with image_list.csv",
        "Sources with labelset.csv",
        "Sources with metadata.csv",
        "Sources with image count match",
        "Sources with empty annotations.csv",
        "Sources with empty image_list.csv",
        "Sources with empty labelset.csv",
        "Sources with empty metadata.csv",
    ],
    "Count": [
        s3_audit.count().execute(),
        s3_audit.filter(_.is_complete).count().execute(),
        s3_audit.filter(_.has_images_folder).count().execute(),
        s3_audit.filter(_.has_annotations_csv).count().execute(),
        s3_audit.filter(_.has_image_list_csv).count().execute(),
        s3_audit.filter(_.has_labelset_csv).count().execute(),
        s3_audit.filter(_.has_metadata_csv).count().execute(),
        s3_audit.filter(_.image_count_match).count().execute(),
        s3_audit.filter(_.annotations_empty).count().execute(),
        s3_audit.filter(_.image_list_empty).count().execute(),
        s3_audit.filter(_.labelset_empty).count().execute(),
        s3_audit.filter(_.metadata_empty).count().execute(),
    ],
}
summary_df = pd.DataFrame(summary_data)
total = summary_df.loc[0, "Count"]
summary_df["Percentage"] = (summary_df["Count"] / total * 100).round(1)

In [26]:
GT(summary_df).tab_header(title="CoralNet Source Audit Summary")

GT(_tbl_data=                                Metric  Count  Percentage
0                        Total sources   1500       100.0
1                     Complete sources    812        54.1
2           Sources with images folder   1461        97.4
3         Sources with annotations.csv    898        59.9
4          Sources with image_list.csv    897        59.8
5            Sources with labelset.csv    902        60.1
6            Sources with metadata.csv    612        40.8
7       Sources with image count match    894        59.6
8   Sources with empty annotations.csv     79         5.3
9    Sources with empty image_list.csv      0         0.0
10     Sources with empty labelset.csv      0         0.0
11     Sources with empty metadata.csv      0         0.0, _body=<great_tables._gt_data.Body object at 0x1539fed80>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='right', column_width=None), ColInfo(var='Percentage', type=<ColInfoTypeEnum.default: 1>, column_label='Percentage', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1540b3710>, _spanners=Spanners([]), _heading=Heading(title='CoralNet Source Audit Summary', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1545afc80>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1545afd70>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1545ae060>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=Tru

In [27]:
totals = s3_audit.aggregate(
    total_images_s3=s3_audit.n_images_s3.sum(),
    total_images_csv=s3_audit.n_images_csv.sum(),
    total_annotations=s3_audit.n_annotations.sum(),
    total_unique_annotated=s3_audit.n_unique_images_annotated.sum(),
    total_labels=s3_audit.n_labels.sum(),
    total_metadata_rows=s3_audit.n_metadata_rows.sum(),
).execute()

totals_df = totals.T.reset_index()
totals_df.columns = ["Metric", "Value"]

In [28]:
GT(totals_df).tab_header(title="Total Resource Counts").fmt_number(columns=["Value"], sep_mark=",", decimals=0)

GT(_tbl_data=                   Metric       Value
0         total_images_s3   1101071.0
1        total_images_csv    507349.0
2       total_annotations  26812746.0
3  total_unique_annotated    903805.0
4            total_labels     42188.0
5     total_metadata_rows      2133.0, _body=<great_tables._gt_data.Body object at 0x1545af7d0>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1545af530>, _spanners=Spanners([]), _heading=Heading(title='Total Resource Counts', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1545ad700>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1545ae7e0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1545ae450>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1545aeba0>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_left_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), heading_background_color=OptionsInfo(scss=True, category='heading', type='value', value=None), heading_align=OptionsInfo(scss=True, category='he

In [7]:
cn_annotations_raw = con.read_parquet(CORALNET_ANNOTATIONS_URI)
cn_for_source = cn_annotations_raw.filter(cn_annotations_raw.source_id == EXAMPLE_SOURCE_ID)

anno_summary = cn_for_source.aggregate(
    n_points=cn_for_source.count(),
    n_images=cn_for_source.image_id.nunique(),
).execute()

In [8]:
cn_annotations_raw

┏━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━┓
┃ source_id ┃ image_id ┃ row   ┃ col   ┃ coralnet_id ┃
┡━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━┩
│ int64     │ string   │ int64 │ int64 │ int64       │
├───────────┼──────────┼───────┼───────┼─────────────┤
│        23 │ 12805    │   735 │  1008 │         112 │
│        23 │ 12805    │   663 │  1682 │         106 │
│        23 │ 12805    │   955 │  1737 │         106 │
│        23 │ 12805    │  1034 │  1431 │         105 │
│        23 │ 12805    │   851 │  2036 │         106 │
│        23 │ 12805    │   833 │  2038 │         100 │
│        23 │ 12805    │  1235 │  1158 │         111 │
│        23 │ 12805    │  1294 │  1312 │         105 │
│        23 │ 12805    │  1194 │  1319 │         105 │
│        23 │ 12805    │  1498 │  3096 │         105 │
│         … │ …        │     … │     … │           … │
└───────────┴──────────┴───────┴───────┴─────────────┘

In [11]:
anno_summary

,n_points,n_images
0,20200,1000


In [7]:
sources_p = cn_annotations_raw.select(_.source_id).distinct()

In [ ]:
LIST_OF_SOURCES = [4957, 5086, 5444, 5847, 6521, 6537, 6687, 6699]

In [ ]:
sources_p.filter(sources_p.source_id.isin(LIST_OF_SOURCES)).execute()